# BioSentinel v2.3 — Longitudinal Biomarker Trend Analysis

This notebook demonstrates BioSentinel's core insight: **a single test result tells you almost nothing. A trend tells you everything.**

We use the included `patient_sample.json` (fully synthetic data) to:
1. Visualise longitudinal biomarker trajectories
2. Compute trend slopes that feed the ML models
3. Show how risk scores evolve over multiple checkups
4. Demonstrate the SHAP feature attribution output

**All data is 100% synthetic. No real patient data is used.**

---
*Author: Mohit Chaprana / Liveupx Pvt. Ltd. | github.com/liveupx/biosentinel*

In [ ]:
import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Matplotlib style
plt.rcParams.update({
    'figure.facecolor': '#0a1628',
    'axes.facecolor': '#0f2240',
    'axes.edgecolor': '#1e3a5f',
    'axes.labelcolor': '#8fb3a0',
    'text.color': '#e8f4f0',
    'xtick.color': '#4a7060',
    'ytick.color': '#4a7060',
    'grid.color': '#1e3a5f',
    'grid.alpha': 0.4,
    'lines.linewidth': 2.0,
    'font.family': 'sans-serif',
})

# Load patient sample — works from any directory
sample_path = os.path.join(os.path.dirname(os.path.abspath('.')), 
                           'biosentinel-main', 'patient_sample.json')
if not os.path.exists(sample_path):
    sample_path = 'patient_sample.json'  # fallback: run from repo root

with open(sample_path) as f:
    patient = json.load(f)

print(f"Patient: {patient['demographics']['age']}yr {patient['demographics']['sex']}")
print(f"Checkups: {len(patient['checkups'])}")
print(f"Medications: {len(patient['medications'])}")
print(f"Date range: {patient['checkups'][0]['date']} → {patient['checkups'][-1]['date']}")

In [ ]:
# ── Build longitudinal dataframe ────────────────────────────────────────────
def _flatten_checkup(chk):
    """Flatten nested lab_results sections into a flat dict."""
    row = {'date': pd.to_datetime(chk['date'])}
    for section in chk.get('lab_results', {}).values():
        if isinstance(section, dict):
            row.update(section)
    vitals = chk.get('vitals', {})
    row.update({'bp_systolic': vitals.get('blood_pressure_systolic'),
                'bmi': vitals.get('bmi')})
    return row

df = pd.DataFrame([_flatten_checkup(c) for c in patient['checkups']])
df = df.set_index('date').sort_index()
print(f"DataFrame: {df.shape[0]} rows x {df.shape[1]} columns")
print("\nKey biomarkers available:")
available = [c for c in df.columns if df[c].notna().sum() > 0]
print(', '.join(available[:20]))
print(f"\nHbA1c trajectory: {df['hba1c'].tolist() if 'hba1c' in df else 'N/A'}")


In [ ]:
# ── Plot 1: Biomarker trajectories with reference ranges ───────────────────
BIOMARKER_PLOTS = [
    ('hba1c',          'HbA1c (%)',          (4.0, 5.7),  '#00e896', 'Pre-diabetic > 5.7%'),
    ('glucose_fasting', 'Fasting Glucose\n(mg/dL)', (70, 99), '#00d4e8', 'Pre-diabetic > 100'),
    ('cea',            'CEA (ng/mL)',         (0,   3.0),  '#ff9f43', 'Elevated > 3.0'),
    ('hemoglobin',     'Hemoglobin (g/dL)',   (12,  17.5), '#ff6b9d', 'Low < 12 (F) / 13.5 (M)'),
    ('lymphocytes_pct','Lymphocytes (%)',     (20,  40),   '#a29bfe', 'Low < 20%'),
    ('ldl',            'LDL (mg/dL)',         (0,   100),  '#fd79a8', 'High > 130'),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('BioSentinel — Longitudinal Biomarker Trajectories\n'
             f'Patient: {patient["demographics"]["age"]}yr '
             f'{patient["demographics"]["sex"].title()}, '
             f'{patient["demographics"]["ethnicity"].replace("_"," ").title()}',
             fontsize=14, fontweight='bold', color='#00e896', y=1.02)

for ax, (field, label, (ref_lo, ref_hi), color, note) in zip(axes.flat, BIOMARKER_PLOTS):
    if field not in df.columns or df[field].notna().sum() == 0:
        ax.text(0.5, 0.5, f'No data\nfor {field}', ha='center', va='center',
                transform=ax.transAxes, color='#4a7060')
        ax.set_title(label.replace('\n', ' '), color='#8fb3a0')
        continue

    series = df[field].dropna()
    dates = series.index

    # Reference range band
    ax.axhspan(ref_lo, ref_hi, alpha=0.08, color='#00e896', zorder=0)
    ax.axhline(ref_hi, color='#00e896', linewidth=0.8, linestyle='--', alpha=0.5)
    ax.axhline(ref_lo, color='#00e896', linewidth=0.8, linestyle='--', alpha=0.5)

    # Trend line
    ax.plot(dates, series.values, color=color, marker='o', markersize=6,
            markerfacecolor='white', markeredgecolor=color, zorder=3)

    # Trend slope annotation
    if len(series) >= 2:
        slope = (series.values[-1] - series.values[0]) / max(len(series)-1, 1)
        direction = '↑' if slope > 0.001 else '↓' if slope < -0.001 else '→'
        slope_color = '#ff6b9d' if direction == '↑' else '#00e896' if direction == '↓' else '#00d4e8'
        ax.annotate(f'{direction} {abs(slope):.2f}/visit',
                    xy=(dates[-1], series.values[-1]),
                    xytext=(10, 5), textcoords='offset points',
                    color=slope_color, fontsize=8, fontweight='bold')

    ax.set_title(label.replace('\n',' '), color='#e8f4f0', fontsize=10, pad=8)
    ax.set_ylabel(label.split('\n')[0], fontsize=8)
    ax.tick_params(axis='x', rotation=30, labelsize=7)
    ax.tick_params(axis='y', labelsize=8)
    ax.grid(True, alpha=0.3)
    ax.text(0.02, 0.97, note, transform=ax.transAxes,
            fontsize=7, color='#4a7060', va='top')

plt.tight_layout()
plt.savefig('img/biomarker_trajectories.png', dpi=150, bbox_inches='tight',
            facecolor='#0a1628')
plt.show()
print('✅ Saved: img/biomarker_trajectories.png')

In [ ]:
# ── Plot 2: Risk score evolution (mock — shows concept) ────────────────────
checkup_labels = [c['date'][:7] for c in patient['checkups']]
n = len(checkup_labels)

# Simulate risk trajectory based on actual biomarker trends
np.random.seed(42)
hba1c_vals = [c['lab_results'].get('hba1c', 5.5) for c in patient['checkups']]
cea_vals   = [c['lab_results'].get('cea', 1.5)   for c in patient['checkups']]

def _score(hba1c, cea):
    m = min(0.05 + (hba1c - 5.4) * 0.12 + max(0, cea - 2.0) * 0.03, 0.85)
    c = min(0.04 + max(0, cea - 1.5) * 0.04, 0.75)
    return m, c

metabolic_risk  = [_score(h, c)[0] + np.random.normal(0, 0.01) for h, c in zip(hba1c_vals, cea_vals)]
cancer_risk     = [_score(h, c)[1] + np.random.normal(0, 0.008) for h, c in zip(hba1c_vals, cea_vals)]
cardio_risk     = [0.12 + i*0.008 + np.random.normal(0, 0.008) for i in range(n)]
hematologic_risk= [0.06 + np.random.normal(0, 0.005) for _ in range(n)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('BioSentinel — Risk Score Evolution Over Time\n'
             '(Illustrative based on synthetic biomarker trends)',
             fontsize=13, fontweight='bold', color='#00e896')

# Line chart
x = range(n)
ax1.plot(x, metabolic_risk,  label='Metabolic',    color='#00e896', marker='o', markersize=5)
ax1.plot(x, cancer_risk,     label='Cancer',       color='#ff9f43', marker='s', markersize=5)
ax1.plot(x, cardio_risk,     label='Cardiovascular',color='#00d4e8', marker='^', markersize=5)
ax1.plot(x, hematologic_risk,label='Hematologic',  color='#a29bfe', marker='D', markersize=5)

# Risk level zones
ax1.axhspan(0,    0.25, alpha=0.05, color='#00e896')
ax1.axhspan(0.25, 0.50, alpha=0.05, color='#fdcb6e')
ax1.axhspan(0.50, 0.75, alpha=0.05, color='#e17055')
ax1.axhspan(0.75, 1.00, alpha=0.05, color='#d63031')
for y, label in [(0.125,'LOW'), (0.375,'MOD'), (0.625,'HIGH'), (0.875,'CRIT')]:
    ax1.text(n-0.5, y, label, fontsize=7, color='#4a7060', ha='right', va='center')

ax1.set_xticks(x)
ax1.set_xticklabels(checkup_labels, rotation=35, fontsize=7)
ax1.set_ylim(0, 1)
ax1.set_ylabel('Risk Score (0–1)', fontsize=9)
ax1.set_title('Risk Trajectory per Domain', color='#e8f4f0')
ax1.legend(fontsize=8, facecolor='#0f2240', edgecolor='#1e3a5f', labelcolor='#e8f4f0')
ax1.grid(True, alpha=0.3)

# Latest snapshot bar chart
domains = ['Cancer', 'Metabolic', 'Cardio', 'Hema']
latest  = [cancer_risk[-1], metabolic_risk[-1], cardio_risk[-1], hematologic_risk[-1]]
colors  = ['#ff9f43' if v >= 0.25 else '#00e896' for v in latest]
bars = ax2.barh(domains, [v*100 for v in latest], color=colors, height=0.5, 
                edgecolor='none', alpha=0.85)
for bar, val in zip(bars, latest):
    level = 'CRITICAL' if val>=0.75 else 'HIGH' if val>=0.50 else 'MODERATE' if val>=0.25 else 'LOW'
    ax2.text(val*100 + 0.5, bar.get_y() + bar.get_height()/2,
             f'{val*100:.1f}%  [{level}]', va='center', fontsize=9, color='#e8f4f0')
ax2.set_xlim(0, 80)
ax2.set_xlabel('Risk Score (%)', fontsize=9)
ax2.set_title(f'Latest Assessment — {checkup_labels[-1]}', color='#e8f4f0')
ax2.grid(True, axis='x', alpha=0.3)
ax2.axvline(25, color='#fdcb6e', linestyle='--', alpha=0.4, linewidth=1)
ax2.axvline(50, color='#e17055', linestyle='--', alpha=0.4, linewidth=1)

plt.tight_layout()
plt.savefig('img/risk_trajectory.png', dpi=150, bbox_inches='tight', facecolor='#0a1628')
plt.show()
print('✅ Saved: img/risk_trajectory.png')

In [ ]:
# ── Trend slope analysis (core BioSentinel signal) ─────────────────────────
print('BioSentinel — Trend Slope Analysis')
print('=' * 55)
print(f'Patient: {patient["demographics"]["age"]}yr '
      f'{patient["demographics"]["sex"].title()}')
print(f'Timeline: {len(patient["checkups"])} checkups')
print()

RISK_MARKERS = {
    'hba1c':           ('HbA1c',          '%',     'up',   5.7),
    'glucose_fasting': ('Fasting Glucose', 'mg/dL', 'up',   99),
    'cea':             ('CEA',             'ng/mL', 'up',   3.0),
    'hemoglobin':      ('Hemoglobin',      'g/dL',  'down', 12.0),
    'lymphocytes_pct': ('Lymphocytes',     '%',     'down', 20.0),
    'ldl':             ('LDL',             'mg/dL', 'up',   130),
    'bp_systolic':     ('BP Systolic',     'mmHg',  'up',   130),
}

alerts = []
print(f'{"Biomarker":<20} {"First":>8} {"Latest":>8} {"Slope/visit":>12} {"Direction":>10} {"Status"}')
print('-' * 75)

for field, (name, unit, concern_dir, threshold) in RISK_MARKERS.items():
    vals = [c['lab_results'].get(field) for c in patient['checkups']]
    vals = [v for v in vals if v is not None]
    if len(vals) < 2:
        continue

    slope = (vals[-1] - vals[0]) / (len(vals) - 1)
    direction = '↑ Rising' if slope > 0.001 else '↓ Falling' if slope < -0.001 else '→ Stable'
    
    latest_val = vals[-1]
    is_concerning = (
        (concern_dir == 'up'   and slope > 0.001 and latest_val > threshold * 0.85) or
        (concern_dir == 'down' and slope < -0.001 and latest_val < threshold * 1.15)
    )
    status = '⚠ TREND ALERT' if is_concerning else '✓ OK'
    if is_concerning:
        alerts.append((name, direction, latest_val, unit, threshold))

    print(f'{name:<20} {vals[0]:>8.2f} {vals[-1]:>8.2f} '
          f'{slope:>12.3f} {direction:>10}   {status}')

print()
if alerts:
    print('🔔 TREND ALERTS DETECTED:')
    for name, direction, val, unit, threshold in alerts:
        print(f'   • {name}: {direction} (current: {val:.1f} {unit}, '
              f'threshold: {threshold} {unit})')
    print()
    print('→ These trends fed into the ML model produce elevated risk scores.')
else:
    print('✅ No trend alerts detected.')

In [ ]:
# ── SHAP feature attribution illustration ──────────────────────────────────
# This shows what the ML engine returns for feature attribution.
# In production, these values come from shap.TreeExplainer.

shap_example = {
    'hba1c_slope':        0.142,   # rising blood sugar trend
    'glucose_slope':      0.098,   # rising glucose
    'cea_slope':          0.071,   # CEA creeping up
    'age':                0.058,   # age factor
    'fam_cancer':         0.045,   # family history
    'lymph_slope':       -0.032,   # lymphocytes declining (protective lost)
    'bmi_slope':          0.028,   # BMI increasing
    'n_high':             0.021,   # reference range violations count
    'hdl':               -0.018,   # HDL protective
    'exercise_inv':       0.012,   # low exercise
}

fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor('#0a1628')
ax.set_facecolor('#0f2240')

items = sorted(shap_example.items(), key=lambda x: x[1])
names = [i[0].replace('_', ' ').title() for i in items]
values = [i[1] for i in items]
colors = ['#00e896' if v < 0 else '#ff6b9d' for v in values]

bars = ax.barh(names, values, color=colors, height=0.6, alpha=0.85)
ax.axvline(0, color='#4a7060', linewidth=1)

for bar, val in zip(bars, values):
    offset = 0.003 if val >= 0 else -0.003
    ha = 'left' if val >= 0 else 'right'
    ax.text(val + offset, bar.get_y() + bar.get_height()/2,
            f'{val:+.3f}', va='center', ha=ha, fontsize=8.5, color='#e8f4f0')

ax.set_xlabel('SHAP Value (impact on metabolic risk score)', color='#8fb3a0', fontsize=10)
ax.set_title('BioSentinel — SHAP Feature Attribution\n'
             'Which biomarker trends are driving the metabolic risk score?',
             color='#e8f4f0', fontsize=12, pad=12)
ax.tick_params(colors='#8fb3a0', labelsize=9)
ax.grid(True, axis='x', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_color('#1e3a5f')
ax.spines['left'].set_color('#1e3a5f')

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#ff6b9d', alpha=0.85, label='Increases risk'),
    Patch(facecolor='#00e896', alpha=0.85, label='Decreases risk (protective)'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9,
          facecolor='#0f2240', edgecolor='#1e3a5f', labelcolor='#e8f4f0')

plt.tight_layout()
plt.savefig('img/shap_attribution.png', dpi=150, bbox_inches='tight', facecolor='#0a1628')
plt.show()
print('✅ Saved: img/shap_attribution.png')
print()
print('Top risk driver: HbA1c slope (rising blood sugar trajectory)')
print('Protective factor: Lymphocyte decline reversed in latest visits')

## Summary

This notebook demonstrated BioSentinel's three-layer analysis:

1. **Trajectory visualisation** — individual biomarkers plotted against reference ranges over time
2. **Trend slope computation** — directional signals (↑↓→) that the ML models use as features
3. **SHAP attribution** — which specific trends drove the risk score, and by how much

The key insight: none of this patient's individual test results crossed a clinical threshold.  
But the *combination of rising trends* — HbA1c creeping up, CEA slowly climbing, glucose on an upward trajectory — produces an elevated metabolic risk score that warrants clinical attention.

This is the core value of longitudinal monitoring over snapshot testing.

---
**Next steps:**
- Run `python run.py` to start the full BioSentinel platform
- Open `biosentinel_dashboard.html` to see the interactive risk assessment UI
- Apply for MIMIC-IV access at physionet.org to retrain models on real data

*⚕ Medical disclaimer: BioSentinel is a research platform, not a licensed medical device.*  
*github.com/liveupx/biosentinel*